# Lab 04. PyTorch 선형회귀: Forward와 Loss

## One Lab, One Notebook

이 파일 하나가 GitHub(깃허브) 원본이자 Colab(코랩) 실습 파일입니다. Ubuntu(우분투)·VS Code(비에스 코드)에서도 같은 파일을 엽니다.

## 목표

- Linear Regression(리니어 리그레션, 선형회귀)이 연속적인 값을 예측하는 방법임을 이해하기
- Hypothesis(하이파서시스, 가설) $H(x)=Wx+b$의 의미 확인하기
- Prediction(프리딕션, 예측값)과 실제값의 차이 계산하기
- MSE(엠에스이, 평균제곱오차)로 Loss(로스, 손실) 계산하기
- Forward(포워드, 순전파) → Loss → Backward(백워드, 역전파) → Update(업데이트, 매개변수 갱신) 흐름 익히기
- 학습된 $W$와 $b$로 새로운 입력값 예측하기

> **A model learns by reducing the difference between prediction and target.**  
> **모델은 예측값과 실제값의 차이를 줄이는 방향으로 학습합니다.**

## 참고 자료와 현행화 기준

- 참고: [Lab02-Linear Regression Google Slides](https://docs.google.com/presentation/d/12raZrY3d244q6jGuC7EykeSPzjP1-FqofMiNlx5Q52o/edit)
- 원본의 TensorFlow 1.x(텐서플로 1.x) `Session(세션)`·`Placeholder(플레이스홀더)` 방식은 사용하지 않습니다.
- 같은 학습 원리를 PyTorch(파이토치)의 동적 계산 그래프와 자동미분 방식으로 구현합니다.


## Setup(셋업, 실행 준비)

### 방법 1. Colab에서 실행

Colab에는 PyTorch가 기본 설치되어 있습니다. 위에서 아래로 `Shift + Enter`를 눌러 실행합니다.

### 방법 2. Ubuntu·VS Code에서 실행

가상환경을 만든 후 필요한 라이브러리를 설치합니다.

```bash
# 가상환경 생성
python3 -m venv .venv

# 가상환경 활성화
source .venv/bin/activate

# PyTorch·Jupyter 설치
python -m pip install torch jupyter
```

VS Code에서 이 파일을 열고 오른쪽 위의 Kernel(커널, 코드를 실행하는 환경) 선택에서 `.venv`를 선택합니다.


In [ ]:
import torch

# 실습 결과를 재현할 수 있도록 난수 시드를 고정합니다.
torch.manual_seed(42)

print("PyTorch 버전:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())


## Steps(스텝스, 실습 단계)

### 1. 선형회귀 문제 이해하기

Linear Regression(리니어 리그레션, 선형회귀)은 입력 $x$로부터 연속적인 값 $y$를 예측합니다.

| 구분 | 의미 | 예 |
|:---|:---|:---|
| Input(인풋, 입력) $x$ | 예측에 사용하는 값 | 공부 시간 |
| Target(타깃, 실제값) $y$ | 모델이 맞혀야 하는 정답 | 시험 점수 |
| Prediction(프리딕션, 예측값) $\hat{y}$ | 모델이 계산한 값 | 예상 시험 점수 |
| Regression(리그레션, 회귀) | 연속적인 수치 예측 | 온도·전압·속도·가격 |

이번 실습은 원리를 분명하게 보기 위해 다음 세 점을 사용합니다.

| $x$ | $y$ |
|:---|:---|
| 1 | 1 |
| 2 | 2 |
| 3 | 3 |


In [ ]:
# 입력 x와 실제값 y를 float32 Tensor로 만듭니다.
x_train = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
y_train = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)

print("x_train:", x_train)
print("y_train:", y_train)
print("shape:", x_train.shape, y_train.shape)


### 2. Hypothesis: 직선으로 관계 표현하기

선형회귀는 데이터의 관계를 다음 직선으로 가정합니다.

$$
H(x)=Wx+b
$$

| 기호 | 읽기와 의미 |
|:---|:---|
| $H(x)$ 또는 $\hat{y}$ | Hypothesis의 출력, 예측값 |
| $W$ | Weight(웨이트, 가중치), 직선의 기울기 |
| $b$ | Bias(바이어스, 편향), 직선의 $y$절편 |
| $x$ | 모델에 입력하는 값 |

$W$가 바뀌면 직선의 기울기가 바뀌고, $b$가 바뀌면 직선이 위아래로 이동합니다.

> **Forward computes predictions from inputs and parameters.**  
> **Forward(포워드, 순전파)는 입력과 매개변수로부터 예측값을 계산합니다.**


In [ ]:
# 아직 학습하지 않고 W와 b를 직접 정해 Forward를 계산합니다.
W_example = torch.tensor(0.5)
b_example = torch.tensor(2.0)

y_pred_example = W_example * x_train + b_example

print("가설: H(x) = 0.5x + 2.0")
print("예측값:", y_pred_example)
print("실제값:", y_train)
print("오차:", y_pred_example - y_train)


### 3. Loss: 예측이 얼마나 틀렸는지 계산하기

MSE(엠에스이, 평균제곱오차)는 각 예측 오차를 제곱한 뒤 평균을 계산합니다.

$$
\mathrm{MSE}=
\frac{1}{m}\sum_{i=1}^{m}
\left(H\left(x^{(i)}\right)-y^{(i)}\right)^2
$$

| 계산 단계 | 역할 |
|:---|:---|
| $H(x)-y$ | 예측값과 실제값의 차이 계산 |
| $(H(x)-y)^2$ | 양수·음수 오차의 상쇄 방지 |
| 평균 | 데이터 한 개당 제곱 오차 계산 |

Loss가 작을수록 현재 직선이 데이터에 더 잘 맞습니다.


In [ ]:
# MSE를 구성하는 과정을 한 단계씩 확인합니다.
errors = y_pred_example - y_train
squared_errors = errors ** 2
mse_manual = squared_errors.mean()

print("오차:", errors)
print("제곱 오차:", squared_errors)
print("MSE:", mse_manual.item())


### 4. 두 Hypothesis의 Loss 비교하기

사람은 그래프를 보고 좋은 직선을 고를 수 있지만, 컴퓨터는 Loss를 숫자로 비교합니다.


In [ ]:
def mse_for_line(x, y, weight, bias):
    # 주어진 W와 b로 예측한 뒤 MSE를 반환합니다.
    prediction = weight * x + bias
    return ((prediction - y) ** 2).mean()


loss_h1 = mse_for_line(x_train, y_train, weight=1.0, bias=0.0)
loss_h2 = mse_for_line(x_train, y_train, weight=0.5, bias=2.0)

print("H1(x) = 1.0x + 0.0 → MSE:", loss_h1.item())
print("H2(x) = 0.5x + 2.0 → MSE:", round(loss_h2.item(), 4))
print("더 좋은 가설:", "H1" if loss_h1 < loss_h2 else "H2")


### 5. 학습할 W와 b 준비하기

PyTorch에서 `requires_grad=True`를 지정하면 해당 Tensor의 Gradient(그레이디언트, 기울기)를 자동으로 계산할 수 있습니다.

초기값은 정답이 아닙니다. 학습을 반복하면서 Loss가 줄어드는 방향으로 $W$와 $b$가 바뀝니다.


In [ ]:
# W와 b는 학습으로 바뀌는 Parameter(파라미터, 매개변수)입니다.
W = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)
b = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

print("초기 W:", W.item())
print("초기 b:", b.item())
print("W의 자동미분 사용 여부:", W.requires_grad)


### 6. Forward → Loss → Backward 한 번 실행하기

학습의 기본 흐름은 다음 네 단계입니다.

| 순서 | PyTorch 코드 | 의미 |
|:---|:---|:---|
| 1 | `y_pred = W * x + b` | Forward(포워드, 예측값 계산) |
| 2 | `loss = ((y_pred - y) ** 2).mean()` | Loss(로스, 오차 크기 계산) |
| 3 | `loss.backward()` | Backward(백워드, 기울기 계산) |
| 4 | `optimizer.step()` | Update(업데이트, W와 b 갱신) |

`backward()`는 Loss를 줄이려면 $W$와 $b$를 어느 방향으로 바꿔야 하는지 계산합니다.


In [ ]:
# 1. Forward: 현재 W와 b로 예측합니다.
y_pred = W * x_train + b

# 2. Loss: MSE를 계산합니다.
loss = ((y_pred - y_train) ** 2).mean()

# 3. Backward: W와 b에 대한 Loss의 기울기를 계산합니다.
loss.backward()

print("예측값:", y_pred.detach())
print("Loss:", loss.item())
print("W.grad:", W.grad.item())
print("b.grad:", b.grad.item())


### 7. Optimizer로 W와 b 학습하기

SGD(에스지디, 확률적 경사하강법)는 기울기의 반대 방향으로 매개변수를 조금씩 이동합니다.

$$
\text{새 매개변수}
\leftarrow
\text{현재 매개변수}
-
\text{learning rate}\times\text{gradient}
$$

Learning Rate(러닝 레이트, 학습률)는 한 번에 이동하는 크기입니다. 이번 실습에서는 `0.01`을 사용합니다.

> **Training repeats forward, loss, backward, and update.**  
> **학습은 Forward → Loss → Backward → Update를 반복합니다.**


In [ ]:
# 앞에서 한 번 계산한 W와 b를 새 Tensor로 초기화합니다.
W = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)
b = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

# SGD Optimizer가 W와 b를 갱신하도록 등록합니다.
optimizer = torch.optim.SGD([W, b], lr=0.01)

loss_history = []

for step in range(2001):
    # 이전 반복에서 계산된 기울기를 0으로 초기화합니다.
    optimizer.zero_grad()

    # Forward: 예측값을 계산합니다.
    y_pred = W * x_train + b

    # Loss: 평균제곱오차를 계산합니다.
    loss = ((y_pred - y_train) ** 2).mean()

    # Backward: W와 b의 기울기를 계산합니다.
    loss.backward()

    # Update: 기울기를 이용해 W와 b를 갱신합니다.
    optimizer.step()

    loss_history.append(loss.item())

    if step % 400 == 0:
        print(
            f"step {step:4d} | "
            f"loss {loss.item():.8f} | "
            f"W {W.item():.4f} | b {b.item():.4f}"
        )


### 8. 학습 결과 해석하기

데이터가 $(1,1)$, $(2,2)$, $(3,3)$이므로 가장 잘 맞는 직선은 $H(x)=1x+0$입니다.

학습이 정상적으로 진행되면 다음 변화가 나타납니다.

- Loss는 0에 가까워집니다.
- $W$는 1에 가까워집니다.
- $b$는 0에 가까워집니다.


In [ ]:
print("학습된 W:", W.item())
print("학습된 b:", b.item())
print("최종 Loss:", loss_history[-1])
print("Loss 감소:", loss_history[0], "→", loss_history[-1])


### 9. 새로운 입력값 예측하기

학습할 때 사용하지 않은 입력값도 학습된 직선에 넣어 예측할 수 있습니다.

예측 단계에서는 기울기 계산이 필요하지 않으므로 `torch.no_grad()`를 사용합니다.


In [ ]:
x_new = torch.tensor([5.0, 2.5, 1.5, 3.5], dtype=torch.float32)

with torch.no_grad():
    y_new = W * x_new + b

print("새 입력:", x_new)
print("예측 결과:", y_new)


### 10. 슬라이드의 새 학습 데이터 적용하기

슬라이드의 두 번째 예제는 $y=x+1.1$에 가까운 데이터를 사용합니다.

| $x$ | $y$ |
|:---|:---|
| 1 | 2.1 |
| 2 | 3.1 |
| 3 | 4.1 |
| 4 | 5.1 |
| 5 | 6.1 |

반복 학습 코드를 함수로 묶으면 데이터가 바뀌어도 같은 절차를 재사용할 수 있습니다.


In [ ]:
def train_linear_regression(x, y, learning_rate=0.01, epochs=3000):
    # 학습할 W와 b를 0에서 시작합니다.
    weight = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)
    bias = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)
    optimizer = torch.optim.SGD([weight, bias], lr=learning_rate)

    for _ in range(epochs):
        optimizer.zero_grad()
        prediction = weight * x + bias
        loss = ((prediction - y) ** 2).mean()
        loss.backward()
        optimizer.step()

    return weight.detach(), bias.detach(), loss.detach()


x_train_2 = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32)
y_train_2 = torch.tensor([2.1, 3.1, 4.1, 5.1, 6.1], dtype=torch.float32)

W_2, b_2, loss_2 = train_linear_regression(x_train_2, y_train_2)

with torch.no_grad():
    prediction_at_7 = W_2 * torch.tensor(7.0) + b_2

print("학습된 W:", W_2.item())
print("학습된 b:", b_2.item())
print("최종 Loss:", loss_2.item())
print("x=7 예측:", prediction_at_7.item())


## Practice(프랙티스, 확인 실습)

센서의 입력 전압과 실제 측정값 사이의 관계를 선형회귀로 보정합니다.

| 입력 전압 $x$ | 실제 측정값 $y$ |
|:---|:---|
| 0.5 | 11 |
| 1.0 | 21 |
| 1.5 | 31 |
| 2.0 | 41 |

아래 코드의 `TODO`를 확인하고 실행합니다.

- 학습 데이터 Tensor 만들기
- 제공된 함수로 $W$와 $b$ 학습하기
- 입력 전압 1.25 V에 대한 측정값 예측하기


In [ ]:
# TODO 1: 입력 전압과 실제 측정값을 Tensor로 만듭니다.
voltage_data = torch.tensor([0.5, 1.0, 1.5, 2.0], dtype=torch.float32)
measured_data = torch.tensor([11.0, 21.0, 31.0, 41.0], dtype=torch.float32)

# TODO 2: 선형회귀 모델을 학습합니다.
sensor_W, sensor_b, sensor_loss = train_linear_regression(
    voltage_data,
    measured_data,
    learning_rate=0.05,
    epochs=4000,
)

# TODO 3: 새로운 입력 전압 1.25 V의 측정값을 예측합니다.
new_voltage = torch.tensor(1.25, dtype=torch.float32)
with torch.no_grad():
    corrected_value = sensor_W * new_voltage + sensor_b

print("sensor W:", sensor_W.item())
print("sensor b:", sensor_b.item())
print("sensor Loss:", sensor_loss.item())
print("1.25 V 예측값:", corrected_value.item())


## Checks(체크스, 실행 확인)

아래 셀에서 오류가 없으면 Lab 04가 정상적으로 완료된 것입니다.


In [ ]:
assert x_train.shape == torch.Size([3])
assert y_pred_example.shape == torch.Size([3])
assert abs(mse_manual.item() - (3.5 / 3.0)) < 1e-6
assert loss_h1.item() < loss_h2.item()
assert loss_history[-1] < loss_history[0]
assert abs(W.item() - 1.0) < 0.02
assert abs(b.item()) < 0.05
assert abs(W_2.item() - 1.0) < 0.02
assert abs(b_2.item() - 1.1) < 0.05
assert abs(corrected_value.item() - 26.0) < 0.2

print("Lab 04 실행 확인 완료")


## 핵심 정리

| 단계 | 핵심 내용 |
|:---|:---|
| 데이터 | 입력 $x$와 실제값 $y$를 Tensor로 준비 |
| Hypothesis | $H(x)=Wx+b$로 예측값 계산 |
| Loss | MSE로 예측값과 실제값의 차이 계산 |
| Backward | 자동미분으로 $W$와 $b$의 기울기 계산 |
| Update | SGD로 $W$와 $b$ 갱신 |
| 반복 학습 | Loss가 감소하도록 네 단계를 반복 |
| Prediction | 학습된 $W$와 $b$로 새로운 입력 예측 |

> **Linear Regression = Forward → Loss → Backward → Update**

## Next Steps(넥스트 스텝스, 다음 단계)

다음 주제는 새 노트북 `Lab_05_Minimizing_Cost.ipynb`로 진행합니다.

- $W$ 변화에 따른 Cost 곡선 확인
- Gradient(그레이디언트, 기울기)의 방향 이해
- Gradient Descent(그레이디언트 디센트, 경사하강법) 직접 구현
- Learning Rate(러닝 레이트, 학습률)가 너무 작거나 클 때의 차이 비교
